# Path Queries with Stardog (LINDAS)

In [1]:
%pip install -q ipywidgets==8.0.4 ipycytoscape networkx nbformat plotly
import ipycytoscape as cy
import networkx as nx
import json
import pprint
import pandas as pd
from pyodide.ffi import to_js
from IPython.display import JSON, HTML
from js import Object, fetch
from io import StringIO

async def path(query_string, address = "https://ld.admin.ch/query"):
    
    # try the Post request with help of JS fetch
    # the creation of the request header is a little bit complicated because it needs to be a 
    # JavaScript JSON that is made within a Python source code
    try:
        resp = await fetch(address,
          method="POST",
          body="query=" + query_string,
          credentials="same-origin",
          headers=Object.fromEntries(to_js({"Content-Type": "application/x-www-form-urlencoded; charset=UTF-8", 
                                            "Accept": "application/sparql-results+json" })),
        )
    except:
        raise RuntimeError("Fetch failed")
    
    
    if resp.ok:
        res = await resp.text()
        return res
    else:
        raise RuntimeError("Response status " + str(resp.status) + ":" + await resp.text())

In [2]:
res = await path("""

PATHS 
START ?x = <https://ld.admin.ch/municipality/version/14062> 
END ?y 
VIA <https://version.link/successor> | <https://version.link/predecessor>


""", "https://test.lindas.admin.ch/query")

res = json.loads(res)
pprint.pprint(res["results"]["bindings"])

[{'x': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/14062'},
  'y': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/11930'}},
 {},
 {'x': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/14062'},
  'y': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/12041'}},
 {},
 {'x': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/14062'},
  'y': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/12401'}},
 {},
 {'x': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/14062'},
  'y': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/13753'}},
 {},
 {'x': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/14062'},
  'y': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/14063'}},
 {},
 {'x': {'type': 'uri',
        'value': 'https://ld.admin.ch

In [3]:
res = await path("""

PREFIX schema: <http://schema.org/>

PATHS 
START ?x = <https://ld.admin.ch/municipality/version/14062> #Berg (TG)
#START ?x = <https://ld.admin.ch/municipality/version/13445> #Rubigen
#START ?x = <https://ld.admin.ch/municipality/version/12612> #Lavertezzo
#START ?x = <https://ld.admin.ch/municipality/version/14091> #Eschlikon
END ?y 
VIA {

    ?x ?p ?y.
    ?x schema:name ?xName.
    ?y schema:name ?yName.
    
    FILTER(?p = <https://version.link/successor> || ?p = <https://version.link/predecessor>)

}


""", "https://test.lindas.admin.ch/query")

res = json.loads(res)

results = []

for entry in res["results"]["bindings"]:
    if "x" in entry:
        if entry["p"]["value"] == "https://version.link/successor":
            results.append(
                {
                    "source": entry["x"]["value"], 
                    "sourceName": entry["xName"]["value"],
                    "targetName": entry["yName"]["value"],
                    "target": entry["y"]["value"]
                }
            )
        else:
            results.append(
                {
                    "source": entry["y"]["value"], 
                    "sourceName": entry["yName"]["value"],
                    "targetName": entry["xName"]["value"],
                    "target": entry["x"]["value"]
                }
            )
            
edge_df = pd.DataFrame(results)
    
display(edge_df)


ids = pd.Series.tolist(edge_df["source"]) + pd.Series.tolist(edge_df["target"])
names = pd.Series.tolist(edge_df["sourceName"]) + pd.Series.tolist(edge_df["targetName"])

node_df = pd.DataFrame(list(zip(ids, names)), columns = ["id", "name"])
node_df.drop_duplicates(inplace=True)

G = nx.from_pandas_edgelist(edge_df, source="source", target="target", create_using=nx.DiGraph())
nx.set_node_attributes(G, node_df.set_index('id').to_dict('index'))


my_style = [
    {
        'selector': 'node',
         'style': {
            'font-family': 'helvetica',
            'font-size': '12px',
            'label': 'data(name)'
         }
    },
    {
        "selector": "edge.directed",
        "style": {
            "curve-style": "bezier",
            "target-arrow-shape": "triangle",
        }
    }
]

muni = cy.CytoscapeWidget()
muni.graph.add_graph_from_networkx(G, directed=True)
muni.set_style(my_style)
muni.set_layout(name = "dagre", rankDir = "LR")
muni
    
#pprint.pprint(res["results"]["bindings"])

,source,sourceName,targetName,target
0,https://ld.admin.ch/municipality/version/11930,Guntershausen b. Birwin.,Berg (TG),https://ld.admin.ch/municipality/version/14062
1,https://ld.admin.ch/municipality/version/12041,Graltshausen,Berg (TG),https://ld.admin.ch/municipality/version/14062
2,https://ld.admin.ch/municipality/version/12401,Mauren,Berg (TG),https://ld.admin.ch/municipality/version/14062
3,https://ld.admin.ch/municipality/version/13753,Berg (TG),Berg (TG),https://ld.admin.ch/municipality/version/14062
4,https://ld.admin.ch/municipality/version/14062,Berg (TG),Berg (TG),https://ld.admin.ch/municipality/version/14063
...,...,...,...,...
67,https://ld.admin.ch/municipality/version/11544,Illighausen,Lengwil,https://ld.admin.ch/municipality/version/14110
68,https://ld.admin.ch/municipality/version/14062,Berg (TG),Berg (TG),https://ld.admin.ch/municipality/version/14063
69,https://ld.admin.ch/municipality/version/14063,Berg (TG),Berg (TG),https://ld.admin.ch/municipality/version/14112
70,https://ld.admin.ch/municipality/version/14110,Lengwil,Berg (TG),https://ld.admin.ch/municipality/version/14112


CytoscapeWidget(cytoscape_layout={'name': 'dagre', 'rankDir': 'LR'}, cytoscape_style=[{'selector': 'node', 'st…

In [5]:
res = await path("""

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>

PATHS 
START ?x = <https://ld.admin.ch/municipality/version/14062>
END ?y 
VIA {

#BIND(vl:successor as ?p)

VALUES ?p {
    vl:successor
    vl:predecessor
}

?x ?p ?y.

?x schema:name ?xName.
?y schema:name ?yName.
}


""", "https://test.lindas.admin.ch/query")

res = json.loads(res)
pprint.pprint(res["results"]["bindings"])

[{'p': {'type': 'uri', 'value': 'https://version.link/predecessor'},
  'x': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/14062'},
  'xName': {'type': 'literal', 'value': 'Berg (TG)'},
  'y': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/11930'},
  'yName': {'type': 'literal', 'value': 'Guntershausen b. Birwin.'}},
 {},
 {'p': {'type': 'uri', 'value': 'https://version.link/predecessor'},
  'x': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/14062'},
  'xName': {'type': 'literal', 'value': 'Berg (TG)'},
  'y': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/12041'},
  'yName': {'type': 'literal', 'value': 'Graltshausen'}},
 {},
 {'p': {'type': 'uri', 'value': 'https://version.link/predecessor'},
  'x': {'type': 'uri',
        'value': 'https://ld.admin.ch/municipality/version/14062'},
  'xName': {'type': 'literal', 'value': 'Berg (TG)'},
  'y': {'type': 'uri',
        'value'